In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
HF_TOKEN = os.environ.get("HF_TOKEN")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser

from pydantic import BaseModel,Field

In [8]:

from langchain_google_genai import ChatGoogleGenerativeAI
llm_model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")#"gemini-2.5-flash")

In [12]:
class QueryResponse(BaseModel):
    desc: str=Field(description="A brief description of topic asked by user")
    pros: str=Field(description="3 bullet points to show pros of the topic")
    cons: str=Field(description="3 bullet points to show cons of the topic")
    conclusion: str=Field(description="one line conclusion about the topic")

parser = PydanticOutputParser(pydantic_object=QueryResponse)
parser


json_parser = JsonOutputParser(pydantic_object=QueryResponse)
json_parser

JsonOutputParser(pydantic_object=<class '__main__.QueryResponse'>)

In [13]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"desc": {"description": "A brief description of topic asked by user", "title": "Desc", "type": "string"}, "pros": {"description": "3 bullet points to show pros of the topic", "title": "Pros", "type": "string"}, "cons": {"description": "3 bullet points to show cons of the topic", "title": "Cons", "type": "string"}, "conclusion": {"description": "one line conclusion about the topic", "title": "Conclusion", "type": "string"}}, "required": ["desc", "pros", "cons", "conclusion"]}
```


In [14]:
print(json_parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [17]:
prompt_txt = """
    Answer the user query and generate response based on the following instructions
    format Instructions:
    {format_instructions}

    Query:
    {query}

"""

prompt = PromptTemplate(
    template = prompt_txt,
    input_variables=["query"],
    partial_variables={"format_instructions":parser.get_format_instructions()}
)

prompt

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"desc": {"description": "A brief description of topic asked by user", "title": "Desc", "type": "string"}, "pros": {"description": "3 bullet points to show pros of the topic", "title": "Pros", "type": "string"}, "cons": {"description": "3 bullet points to show cons of the topic", "title": "Cons", "type": "string"}, "conclusion": {"description": "one line conclusion about the topic", "title": "Conclusion", "type": "string"}

In [19]:
chain = (
    prompt
    |
    llm_model
    |
    parser
)

response = chain.invoke({"query": "Tell me about the stock market"})
response

QueryResponse(desc='The stock market is a centralized marketplace where shares of publicly held companies are issued, bought, and sold, allowing investors to own a portion of a business and companies to raise capital for growth.', pros='• Potential for high long-term returns that typically outperform inflation.\n• High liquidity allows investors to quickly buy or sell assets for cash.\n• Offers a wide variety of investment options for portfolio diversification.', cons='• Market volatility can lead to significant and sudden financial losses.\n• Success often requires significant research, time, and emotional discipline.\n• Performance is heavily influenced by external factors like economic cycles and geopolitical events.', conclusion='The stock market is a powerful vehicle for wealth creation, provided that investors maintain a long-term perspective and manage their risk exposure.')

In [20]:
chain = (
    prompt
    |
    llm_model
    |
    json_parser
)

response = chain.invoke({"query": "Tell me about the stock market"})
response

{'desc': 'The stock market is a public marketplace where shares of publicly held companies are issued, bought, and sold, allowing investors to own a portion of a corporation and providing businesses with access to capital.',
 'pros': '- Potential for significant long-term wealth accumulation through capital appreciation.\n- Provides a source of passive income through regular dividend payments.\n- Offers high liquidity, allowing investors to convert shares into cash relatively quickly.',
 'cons': '- Prices can be highly volatile, leading to the risk of substantial financial loss.\n- Market performance is influenced by unpredictable economic, political, and global events.\n- Requires time, research, and a degree of financial literacy to manage risks effectively.',
 'conclusion': 'The stock market remains one of the most effective tools for building long-term wealth, provided investors maintain a diversified portfolio and a disciplined approach.'}

In [21]:
queries = ["Tell me about delhi", "Tell me about anime"]
formated_query = [{"query":query} for query in queries]

formated_query

[{'query': 'Tell me about delhi'}, {'query': 'Tell me about anime'}]

In [23]:
response = chain.map().invoke(formated_query)
response

[{'desc': "Delhi, the capital territory of India, is a massive metropolitan area in the north of the country that serves as the nation's political hub and a major historical and cultural center.",
  'pros': "- Rich historical heritage featuring numerous UNESCO World Heritage sites like the Red Fort and Humayun's Tomb.\n- A world-class culinary scene offering everything from legendary street food to diverse international cuisines.\n- An extensive and efficient metro rail network that provides excellent connectivity across the city and neighboring regions.",
  'cons': '- Chronic air pollution issues, particularly severe during the winter months, leading to health concerns.\n- High levels of traffic congestion and overcrowding due to the dense population and urban sprawl.\n- Extreme weather conditions, characterized by scorching summer heatwaves and very cold winter temperatures.',
  'conclusion': 'Delhi is a vibrant city that offers a unique blend of ancient history and modern energy, th